# Music Genre Classification — Training Notebook

This notebook trains a CNN model for music genre classification using Mel spectrograms from the GTZAN dataset.

**Steps:**
1. Clone repository & install dependencies
2. Download GTZAN dataset
3. Preprocess audio → Mel spectrograms
4. Train CNN model (GPU recommended)
5. Evaluate & download trained model

**Runtime:** Go to `Runtime → Change runtime type → GPU` before running.

## 1. Setup

In [ ]:
# Clone the repository (replace with your team's repo URL)
!git clone https://github.com/YOUR_TEAM/music_genre_prediction.git
%cd music_genre_prediction

In [ ]:
# Install dependencies (TensorFlow is pre-installed on Colab)
!pip install -r requirements-colab.txt

In [ ]:
# Verify GPU is available
import tensorflow as tf
print(f"TensorFlow version: {tf.__version__}")
print(f"GPU devices: {tf.config.list_physical_devices('GPU')}")
if not tf.config.list_physical_devices('GPU'):
    print("WARNING: No GPU detected. Go to Runtime -> Change runtime type -> GPU")

## 2. Download GTZAN Dataset

**Option A:** Use Kaggle CLI (upload your `kaggle.json` first).

**Option B:** Download manually from [Kaggle](https://www.kaggle.com/datasets/andradaolteanu/gtzan-dataset-music-genre-classification) and upload to Colab.

In [ ]:
# Option A: Upload kaggle.json and download via CLI
# Uncomment the lines below if using Kaggle CLI

# from google.colab import files
# files.upload()  # Upload kaggle.json
# !mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
# !pip install kaggle

!python setup_dataset.py

## 3. Preprocess Audio

This converts all audio files into Mel spectrograms. Takes ~5-10 minutes.

In [ ]:
!python preprocess.py

In [ ]:
# Inspect preprocessed data
import numpy as np
import matplotlib.pyplot as plt
import config

X = np.load("spectrograms/X.npy")
y = np.load("spectrograms/y.npy")
print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"Labels range: {y.min()} - {y.max()}")
print(f"Samples per genre: {np.bincount(y)}")

# Display a sample spectrogram from each genre
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
for i, ax in enumerate(axes.flat):
    idx = np.where(y == i)[0][0]
    ax.imshow(X[idx, :, :, 0], aspect="auto", origin="lower", cmap="magma")
    ax.set_title(config.GENRES[i])
    ax.set_xlabel("Time")
    ax.set_ylabel("Mel Band")
plt.suptitle("Sample Mel Spectrograms by Genre", fontsize=14)
plt.tight_layout()
plt.show()

## 4. Train the Model

Training with file-level stratified split. Uses EarlyStopping, so it may finish before 30 epochs.

In [ ]:
!python train.py

## 5. Evaluate

In [ ]:
!python evaluate.py

In [ ]:
# Display plots inline
from IPython.display import Image, display

print("Training Curves:")
display(Image("plots/training_curves.png"))

print("\nConfusion Matrix:")
display(Image("plots/confusion_matrix.png"))

## 6. Download Trained Model

Download the model file to your local machine for deployment.

In [ ]:
# Download model to local machine
from google.colab import files
files.download("models/genre_cnn.keras")

# Also download the training history and plots
files.download("models/history.json")
files.download("plots/training_curves.png")
files.download("plots/confusion_matrix.png")

## Next Steps

1. **Upload model to Hugging Face Hub** for Streamlit Cloud deployment:
   ```bash
   pip install huggingface_hub
   huggingface-cli login
   huggingface-cli upload your-team/music-genre-cnn models/genre_cnn.keras genre_cnn.keras
   ```

2. **Test locally**: Place `genre_cnn.keras` in the `models/` folder and run `streamlit run app.py`

3. **Deploy**: Connect your GitHub repo to [Streamlit Community Cloud](https://share.streamlit.io)